# 🎮 Game NPC Voice Foundry MVP — Qwen3-TTS
Generate consistent, distinct NPC character voices for games, animations, and tabletop RPGs.

Use cases: indie games, tabletop RPG, animation, VTubing, interactive fiction.

In [ ]:
!pip install -q qwen-tts soundfile
import os
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
import torch
from qwen_tts import Qwen3TTSModel

In [ ]:
NPC_ROSTER = {
    "Eldrin": {
        "prompt": "deep, elderly, measured, British wise wizard",
        "sample": "A wizard is never late, nor is he early. He arrives precisely when he means to.",
        "emotion_modifiers": {"neutral": "measured, calm", "angry": "fierce, thundering voice", "whispering": "hushed, secretive"}
    },
    "Zix": {
        "prompt": "nasally, excitable, fast-talking, wheedling goblin merchant",
        "sample": "Shinies! You want the shinies? Zix has the best deals in all the realms!",
        "emotion_modifiers": {}
    },
    "Commander Mira": {
        "prompt": "firm, direct, no-nonsense military leader, strong and stoic",
        "sample": "Hold the line! We don't fall back until I give the order.",
        "emotion_modifiers": {"neutral": "firm, direct", "angry": "shouting, commanding", "whispering": "tense, tactical whisper", "triumphant": "proud, victorious shout"}
    },
    "ARIA-7": {
        "prompt": "flat, synthesized, subtly uncanny, robotic AI voice",
        "sample": "System diagnostic complete. Human error detected. Correction protocol initiated.",
        "emotion_modifiers": {}
    },
    "Otto": {
        "prompt": "jovial, gravelly, warm, loud tavern barkeep",
        "sample": "Welcome friend! Pull up a chair, first round of ale is on the house!",
        "emotion_modifiers": {}
    },
    "Shadow Rogue": {
        "prompt": "breathy, sibilant whisper, conspiratorial, edgy",
        "sample": "They won't see me coming until it's too late.",
        "emotion_modifiers": {}
    }
}
OUTPUT_DIR = "npc_audio"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
MODEL_ID = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = Qwen3TTSModel.from_pretrained(
    MODEL_ID, 
    device_map=device, 
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, 
    attn_implementation="sdpa"
)
print("VoiceDesign model loaded!")

In [ ]:
print("Generating NPC Roster...")
for name, data in NPC_ROSTER.items():
    print(f"\n--- Voice Card: {name} ---")
    print(f"Prompt: {data['prompt']}")
    print(f"Sample: {data['sample']}")
    
    audio, sr = model.generate_voice_design(
        text=data["sample"],
        language="English",
        instruct=data["prompt"]
    )
    
    path = f"{OUTPUT_DIR}/npc_{name.lower().replace(' ', '_')}_sample.wav"
    sf.write(path, audio, sr)
    display(Audio(path))

In [ ]:
name = "Commander Mira"
emotions = ["neutral", "angry", "whispering", "triumphant"]
print(f"Generating Emotion Variations for {name}...")

for em in emotions:
    print(f"\nEmotion: {em.upper()}")
    emo_prompt = NPC_ROSTER[name]["prompt"] + ", " + NPC_ROSTER[name]["emotion_modifiers"][em]
    audio, sr = model.generate_voice_design(
        text=NPC_ROSTER[name]["sample"],
        language="English",
        instruct=emo_prompt
    )
    path = f"{OUTPUT_DIR}/npc_mira_{em}.wav"
    sf.write(path, audio, sr)
    display(Audio(path))

In [ ]:
USER_NPC = "Eldrin"
USER_DIALOGUE = [
    "Ah, traveler! You've arrived just in time.",
    "The dark forces are gathering in the eastern woods.",
    "Take this staff. It will light your way in the shadows."
]

print(f"Generating Quest Dialogue for {USER_NPC}...")
npc_prompt = NPC_ROSTER[USER_NPC]["prompt"]

for i, line in enumerate(USER_DIALOGUE):
    print(f"Line {i+1}: {line}")
    audio, sr = model.generate_voice_design(line, "en", npc_prompt)
    path = f"{OUTPUT_DIR}/npc_{USER_NPC.lower()}_quest_start_{i+1:02d}.wav"
    sf.write(path, audio, sr)
    display(Audio(path))

In [ ]:
BASE_MODEL_ID = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
base_model = Qwen3TTSModel.from_pretrained(
    BASE_MODEL_ID, 
    device_map=device, 
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, 
    attn_implementation="sdpa"
)
print("Base model loaded for cloning!")

# Clone Commander Mira's sample
mira_ref_audio = f"{OUTPUT_DIR}/npc_commander_mira_sample.wav"
mira_ref_text = NPC_ROSTER["Commander Mira"]["sample"]
mira_clone_prompt = base_model.create_voice_clone_prompt(mira_ref_audio, mira_ref_text)

new_lines = [
    "Report to the barracks immediately.",
    "We have a breach in sector 4!"
]

for i, line in enumerate(new_lines):
    print(f"Clone Line {i+1}: {line}")
    audio, sr = base_model.generate_voice_clone(
        text=line,
        language="English",
        voice_clone_prompt=mira_clone_prompt
    )
    path = f"{OUTPUT_DIR}/npc_mira_clone_{i+1}.wav"
    sf.write(path, audio, sr)
    display(Audio(path))

## 📤 Exporting for Game Engines

**File Naming Convention:**
- Uses `npc_<character_name>_<event>_<id>.wav` format, which is easily parsable by most game engines.

**Importing to Engines:**
- **Unity:** Drag and drop the `.wav` files into your `Assets` folder. Make sure to check 'Force To Mono' if these are spatialized 3D voices.
- **Godot:** Import into your `res://` directory. Godot handles WAV natively for sound effects and dialogue.
- **Unreal Engine:** Drag into the Content Browser. They will be imported as Sound Waves. Right-click to create Sound Cues for random variations.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("npc_audio", "zip", OUTPUT_DIR)
print("Downloading npc_audio.zip...")
files.download("npc_audio.zip")